# Phase 3 — Day 1 (Kaggle T4)

Same pattern as your existing `kaggle-diffusion.ipynb`:
1. pip install
2. git clone the repo
3. auto-find the dataset paths in `/kaggle/input/`
4. run everything (LPIPS-enabled headline eval at 5 + 20 steps, T4 efficiency, Figures 1 + 3)
5. zip the outputs

**Before you run this notebook**, see `KAGGLE_HOWTO.md` in the repo. The two requirements are:
- Push your latest local commits to GitHub so the clone in Cell 2 has the new files.
- Attach your LOL eval15, LOL-v2, and checkpoint datasets via the Kaggle right panel ("Add Data").

In [ ]:
# === Cell 1: install ===
!pip install -q --upgrade pip
!pip install -q scikit-image lpips fvcore matplotlib

In [ ]:
# === Cell 2: clone repo (same as your existing notebook) ===
import os, sys, shutil, subprocess

BRANCH = "main"
REPO_URL = "https://github.com/chirana07/Diffusion_new_final.git"
REPO_DIR = "/kaggle/working/Diffunet"

if os.path.exists(REPO_DIR):
    shutil.rmtree(REPO_DIR)
subprocess.run(
    ["git", "clone", "--branch", BRANCH, "--single-branch", REPO_URL, REPO_DIR],
    check=True,
)
sys.path.insert(0, REPO_DIR)
os.chdir(REPO_DIR)

print("Repo at:", REPO_DIR)
print("\nKey new files we need:")
for f in ["evaluation.py", "measure_efficiency.py", "make_step_ablation_figure.py",
          "make_teaser_figure.py", "diffusion.py"]:
    p = os.path.join(REPO_DIR, f)
    print(f"  {'OK ' if os.path.exists(p) else 'MISSING'}  {f}")
print("\nIf any are MISSING: you forgot to push the latest commit. Run `git push` locally and re-run this cell.")

In [ ]:
# === Cell 3: auto-discover dataset + checkpoint paths ===
# Searches /kaggle/input/ for things that look like LOL eval15, LOL-v2 Real, and a .pth checkpoint.
# If auto-discovery picks the wrong path, just override the variables manually below.
import glob, os

def find_one(patterns, label):
    hits = []
    for pat in patterns:
        hits.extend(glob.glob(pat, recursive=True))
    hits = sorted(set(hits))
    if not hits:
        print(f"  [{label}] NOT FOUND. Searched: {patterns}")
        return None
    if len(hits) > 1:
        print(f"  [{label}] multiple candidates, picking the first:")
        for h in hits:
            print(f"      {h}")
    print(f"  [{label}] = {hits[0]}")
    return hits[0]

EVAL15_LOW  = find_one(["/kaggle/input/**/eval15/low", "/kaggle/input/**/eval15/Low"],   "EVAL15_LOW")
EVAL15_HIGH = find_one(["/kaggle/input/**/eval15/high", "/kaggle/input/**/eval15/High"], "EVAL15_HIGH")
LOLV2_LOW   = find_one(["/kaggle/input/**/LOL-v2/Real_captured/Test/Low",
                         "/kaggle/input/**/Real_captured/Test/Low"], "LOLV2_LOW")
LOLV2_HIGH  = find_one(["/kaggle/input/**/LOL-v2/Real_captured/Test/Normal",
                         "/kaggle/input/**/Real_captured/Test/Normal"], "LOLV2_HIGH")

ckpt_hits = sorted(glob.glob("/kaggle/input/**/*.pth", recursive=True))
ckpt_hits = [c for c in ckpt_hits if "final" in c.lower() or "best" in c.lower() or "last" in c.lower()] or ckpt_hits
CHECKPOINT = ckpt_hits[0] if ckpt_hits else None
print(f"  [CHECKPOINT] = {CHECKPOINT}")

# === MANUAL OVERRIDE if auto-discovery picked wrong ===
# Uncomment and fix any of these if they're None or wrong:
# EVAL15_LOW  = "/kaggle/input/lol-dataset/eval15/low"
# EVAL15_HIGH = "/kaggle/input/lol-dataset/eval15/high"
# LOLV2_LOW   = "/kaggle/input/lol-v2/LOL-v2/Real_captured/Test/Low"
# LOLV2_HIGH  = "/kaggle/input/lol-v2/LOL-v2/Real_captured/Test/Normal"
# CHECKPOINT  = "/kaggle/input/lumidiff-checkpoint/final.pth"

missing = [n for n, v in [
    ("EVAL15_LOW", EVAL15_LOW), ("EVAL15_HIGH", EVAL15_HIGH),
    ("LOLV2_LOW", LOLV2_LOW), ("LOLV2_HIGH", LOLV2_HIGH),
    ("CHECKPOINT", CHECKPOINT)] if not v]
if missing:
    raise SystemExit(
        f"Missing: {missing}. Either attach the dataset via 'Add Data' on the right "
        "panel, or set the variable manually in this cell."
    )

EVAL_RESULTS = "/kaggle/working/eval_results"
os.makedirs(EVAL_RESULTS, exist_ok=True)
print("\nAll paths OK. Outputs will land in", EVAL_RESULTS)

In [ ]:
# === Cell 4: GPU sanity ===
import torch
print("torch:", torch.__version__, "  cuda:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("device:", torch.cuda.get_device_name(0))
else:
    print("\nNO GPU DETECTED. Set Accelerator -> GPU T4 in the Kaggle right panel "
          "(Settings -> Accelerator), then re-run this notebook.")

In [ ]:
# === Cell 5: Job 1 — LPIPS-enabled headline eval at 5 and 20 steps ===
import sys, subprocess

SPLITS = [
    f"eval15:{EVAL15_LOW}:{EVAL15_HIGH}",
    f"lolv2_real:{LOLV2_LOW}:{LOLV2_HIGH}",
]

for STEPS in (5, 20):
    cmd = [
        sys.executable, "evaluation.py",
        "--splits", *SPLITS,
        "--inference-steps", str(STEPS),
        "--sampler", "ddim",
        "--checkpoint", CHECKPOINT,
        "--results-root", os.path.join(EVAL_RESULTS, "headline"),
        "--tag", f"s{STEPS}",
    ]
    print("$ " + " ".join(cmd))
    subprocess.run(cmd, check=True)

In [ ]:
# === Cell 6: Job 2 — T4 efficiency benchmark ===
cmd = [
    sys.executable, "measure_efficiency.py",
    "--steps", "5", "10", "20",
    "--resolution", "400", "600",
    "--device", "cuda",
    "--checkpoint", CHECKPOINT,
    "--out-csv", os.path.join(EVAL_RESULTS, "efficiency_t4.csv"),
]
print("$ " + " ".join(cmd))
subprocess.run(cmd, check=True)

In [ ]:
# === Cell 7: Job 3a — step ablation 5/10/20/50/100 (DDIM only, no LPIPS for speed) ===
for N in (5, 10, 20, 50, 100):
    cmd = [
        sys.executable, "evaluation.py",
        "--splits", *SPLITS,
        "--inference-steps", str(N),
        "--sampler", "ddim",
        "--checkpoint", CHECKPOINT,
        "--results-root", os.path.join(EVAL_RESULTS, "step_ablation_full"),
        "--tag", f"ddim_s{N}",
        "--no-lpips",
    ]
    print("$ " + " ".join(cmd))
    subprocess.run(cmd, check=True)

In [ ]:
# === Cell 8: Job 4 — Figure 1 (teaser) and Figure 3 (step ablation) ===
subprocess.run([
    sys.executable, "make_teaser_figure.py",
    "--pred-s5",  os.path.join(EVAL_RESULTS, "headline_s5/lolv2_real"),
    "--pred-s20", os.path.join(EVAL_RESULTS, "headline_s20/lolv2_real"),
    "--low",      LOLV2_LOW,
    "--gt",       LOLV2_HIGH,
    "--per-image-csv",     os.path.join(EVAL_RESULTS, "headline_s5/per_image.csv"),
    "--per-image-csv-s20", os.path.join(EVAL_RESULTS, "headline_s20/per_image.csv"),
    "--split", "lolv2_real",
    "--out",     os.path.join(EVAL_RESULTS, "figure1_teaser.pdf"),
    "--out-png", os.path.join(EVAL_RESULTS, "figure1_teaser.png"),
], check=True)

subprocess.run([
    sys.executable, "make_step_ablation_figure.py",
    "--eval-root", EVAL_RESULTS,
    "--out",     os.path.join(EVAL_RESULTS, "figure3_step_ablation.pdf"),
    "--out-png", os.path.join(EVAL_RESULTS, "figure3_step_ablation.png"),
], check=True)

In [ ]:
# === Cell 9: print headline table + zip outputs ===
import csv, shutil
rows = []
for tag in ("s5", "s20"):
    p = os.path.join(EVAL_RESULTS, f"headline_{tag}/summary.csv")
    if os.path.exists(p):
        with open(p) as f:
            rows.extend(csv.DictReader(f))

print("### Headline (paste into Table 1)\n")
print("| Split | Sampler | Steps | PSNR | SSIM | LPIPS | Latency/img (s) |")
print("|---|---|---|---|---|---|---|")
for r in rows:
    lp = r.get('lpips_mean')
    lp_s = f"{float(lp):.4f}" if lp not in (None, '', 'None') else '-'
    print(f"| {r['split']} | {r['sampler']} | {r['inference_steps']} | "
          f"{float(r['psnr_mean']):.3f} | {float(r['ssim_mean']):.4f} | {lp_s} | "
          f"{float(r['runtime_mean']):.3f} |")

print("\n### T4 efficiency\n")
with open(os.path.join(EVAL_RESULTS, "efficiency_t4.csv")) as f:
    print(f.read())

out_zip = "/kaggle/working/phase3_day1_outputs.zip"
if os.path.exists(out_zip):
    os.remove(out_zip)
shutil.make_archive(out_zip.replace('.zip', ''), 'zip', EVAL_RESULTS)
print("\nWrote", out_zip)
print("Download it from the Kaggle right panel (Output tab).")